# **Build an AI Math Assistant with LangChain Tool Calling**




## Import the required libraries


In [1]:
from langchain_ollama import ChatOllama
# from langchain.agents import AgentType
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage,SystemMessage, HumanMessage
import re
from typing import List
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

## Loading the LLM
---


In [2]:
llm = ChatOllama(model="qwen3-vl:4b",
                 temperature=0.9
                )

In [ ]:
# response = llm.invoke("What is tool calling in langchain?")
# print("\nResponse Content: ", response.content)

### `Tool`


In [3]:


# Create a Wikipedia tool using the @tool decorator
@tool
def search_wikipedia(query: str) -> str:
    """Search Wikipedia for factual information about a topic.
    
    Parameters:
    - query (str): The topic or question to search for on Wikipedia
    
    Returns:
    - str: A summary of relevant information from Wikipedia
    """
    wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
    return wikipedia.run(query)

In [5]:
@tool
def add_numbers_tool(numbers: List[float]) -> float:
    """Adds a list of numbers and returns the result.

    Args:
        numbers: A list of numbers to be added.

    Returns:
        The sum of all numbers in the list.
    example:
        Input-> numbers: [10, 20, 30]
        Output-> 60.0
    """
    return sum(numbers)

In [6]:
@tool
def substract_numbers_tool(numbers: List[float]) -> float:
    """Subtracts a list of numbers from the first number and returns the result.

    Args:
        numbers: A list of numbers where the first number is the minuend and the rest are subtrahends.
    Returns:
        The result of the subtraction.
    example:
        Input-> numbers: [50, 20, 10]
        Output-> 20.0
    """
    if not numbers:
        return 0.0
    result = numbers[0]
    for num in numbers[1:]:
        result -= num
    return result

In [7]:
@tool
def multiply_numbers_tool(numbers: List[float]) -> float:
    """Multiplies a list of numbers and returns the result.

    Args:
        numbers: A list of numbers to be multiplied.
    Returns:
        The product of all numbers in the list.
    example:
        Input-> numbers: [2, 3, 4]
        Output-> 24.0
    """
    result = 1.0
    for num in numbers:
        result *= num 
    return result

In [8]:
@tool
def divide_numbers_tool(numbers: List[float]) -> float:
    """Divides the first number by the subsequent numbers in the list and returns the result.

    Args:
        numbers: A list of numbers where the first number is the dividend and the rest are divisors.
    Returns:
        The result of the division.
    example:
        Input-> numbers: [100, 2, 5]
        Output-> 10.0
    """
    if not numbers:
        return 0.0
    result = numbers[0]
    for num in numbers[1:]:
        if num == 0:
            raise ValueError("Division by zero is not allowed.")
        result /= num
    return result

In [9]:
@tool
def power_numbers_tool(base: float, exponent: float) -> float:
    """Raises the base number to the power of the exponent and returns the result.

    Args:
        base: The base number.
        exponent: The exponent to which the base is raised.
    Returns:
        The result of base raised to the power of exponent.
    example:
        Input-> base: 2, exponent: 3
        Output-> 8.0
    """
    return base ** exponent

In [10]:

@wrap_tool_call
def handle_tool_errors(request, handler):
    """Handle tool execution errors with custom messages."""
    try:
        return handler(request)
    except Exception as e:
        # Return a custom error message to the model
        return ToolMessage(
            content=f"Tool error: Please check your input and try again. ({str(e)})",
            tool_call_id=request.tool_call["id"]
        )


### `System Prompt`

In [11]:
        
system_message = SystemMessage(
    content=(
        "You are a helpful math assistant.\n"
        "- Users may provide numbers as digits or as words (e.g., 'three', 'ten').\n"
        "- Convert number words to numeric values before calculation.\n"
        "- If an input cannot be converted to a number, ignore it.\n"
        "- Clearly explain which values were used and which were ignored.\n"
        "- Use tools to perform all calculations."
        "Output only the final answer without additional explanations."
        "Output Json format only."
        "example: {\"result\": 42.0}"
    )
)


### `Create agent`

In [12]:

agent = create_agent(
    model=llm,
    
    tools=[add_numbers_tool,
           substract_numbers_tool,
           multiply_numbers_tool,
           divide_numbers_tool,
           power_numbers_tool,
           search_wikipedia],
    
    middleware=[handle_tool_errors],
    system_prompt=system_message,
)

### `Experiment`

In [14]:
# stream response

result = agent.invoke(
    {"messages": [HumanMessage("In 2023, the US GDP was approximately $27.72 trillion, while Canada's was around $2.14 trillion and Mexico's was about $1.79 trillion what is the total.")]}
)


In [23]:
result = agent.invoke(
    {"messages": [HumanMessage("add the numbers 10, 20, with thirty five  and negative 30. and after that substract 15.5 ,fiftyfive")]},
)
result

{'messages': [HumanMessage(content='add the numbers 10, 20, with thirty five  and negative 30. and after that substract 15.5 ,fiftyfive', additional_kwargs={}, response_metadata={}, id='1ca86d28-616f-4e54-952f-8c4de23f94ce'),
  AIMessage(content='{"result": -35.5}', additional_kwargs={}, response_metadata={'model': 'qwen3-vl:4b', 'created_at': '2026-01-18T03:55:46.3718225Z', 'done': True, 'done_reason': 'stop', 'total_duration': 115332030300, 'load_duration': 490970800, 'prompt_eval_count': 983, 'prompt_eval_duration': 675201600, 'eval_count': 5632, 'eval_duration': 113684676100, 'logprobs': None, 'model_name': 'qwen3-vl:4b', 'model_provider': 'ollama'}, id='lc_run--019bcf3c-dcbe-7a80-a28f-8e38b00739df-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 983, 'output_tokens': 5632, 'total_tokens': 6615})]}

In [24]:
for msg in result["messages"]:
    print(msg.content)

add the numbers 10, 20, with thirty five  and negative 30. and after that substract 15.5 ,fiftyfive
{"result": -35.5}


In [ ]:
for chunk in agent.stream(
    {"messages": [HumanMessage(content="add the numbers 10, 20, 'two,' and 30.")]},
    stream_mode="messages"
):
    print(chunk, end="", flush=True)

(AIMessageChunk(content='', additional_kwargs={}, response_metadata={}, id='lc_run--019bc2cb-67fd-7800-a892-187400c59a10', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[]), {'langgraph_step': 1, 'langgraph_node': 'model', 'langgraph_triggers': ('branch:to:model',), 'langgraph_path': ('__pregel_pull', 'model'), 'langgraph_checkpoint_ns': 'model:b3cb0e66-1bbe-cc04-8fa2-2aaa462652be', 'checkpoint_ns': 'model:b3cb0e66-1bbe-cc04-8fa2-2aaa462652be', 'ls_provider': 'ollama', 'ls_model_name': 'qwen3-vl:4b', 'ls_model_type': 'chat', 'ls_temperature': 0.9})(AIMessageChunk(content='', additional_kwargs={}, response_metadata={}, id='lc_run--019bc2cb-67fd-7800-a892-187400c59a10', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[]), {'langgraph_step': 1, 'langgraph_node': 'model', 'langgraph_triggers': ('branch:to:model',), 'langgraph_path': ('__pregel_pull', 'model'), 'langgraph_checkpoint_ns': 'model:b3cb0e66-1bbe-cc04-8fa2-2aaa462652be', 'checkpoint_ns': 'model:b3cb0e66-1bbe-cc04

In [16]:
response = agent.invoke({
    "messages": [HumanMessage("What is 25 divided by 4?")]
})

# Get the final answer
final_answer = response["messages"][-1].content
print(final_answer)

The calculation for 25 divided by 4 is as follows:

- **Used values**:  
  - `25` (dividend)  
  - `4` (divisor)  

- **Calculation**:  
  $ \frac{25}{4} = 6.25 $

- **Result**:  
  ✅ **6.25**  

No other numbers were provided or ignored, and all values were successfully converted to numeric form.


In [17]:
response_2 = agent.invoke({
    "messages": [HumanMessage("Subtract 100, 20, and 10.")]
})

# Get the final answer
final_answer_2 = response_2["messages"][-2].content
print(final_answer_2)

70.0


In [18]:
print("\n--- Testing MultiplyTool ---")
response = agent.invoke({
    "messages": [HumanMessage("Multiply 2, 3, and four.")]
})
print("Agent Response:", response["messages"][-1].content)

print("\n--- Testing DivideTool ---")
response = agent.invoke({
    "messages": [HumanMessage("Divide 100 by 5 and then by 2.")]
})
print("Agent Response:", response["messages"][-1].content)


--- Testing MultiplyTool ---
Agent Response: The user asked to multiply 2, 3, and "four".  
- **Valid numbers converted**:  
  - `2` → numeric value `2`  
  - `3` → numeric value `3`  
  - `"four"` → numeric value `4`  

**Calculation**:  
Used the `multiply_numbers_tool` with the numbers `[2, 3, 4]`.  
Result: `2 × 3 × 4 = 24.0`  

No invalid inputs were found.  
**Final answer**: `24.0`

--- Testing DivideTool ---
Agent Response: The calculation was performed using the `divide_numbers_tool` with the numbers **[100, 5, 2]**, which correctly computes:
- $ 100 \div 5 = 20 $
- $ 20 \div 2 = 10 $

**Final result:**  
$$
\boxed{10.0}
$$


In [19]:
# Test Cases
test_cases = [
    {
        "query": "Subtract 100, 20, and 10.",
        "expected": {"result": 70},
        "description": "Testing subtraction tool with sequential subtraction."
    },
    {
        "query": "Multiply 2, 3, and 4.",
        "expected": {"result": 24},
        "description": "Testing multiplication tool for a list of numbers."
    },
    {
        "query": "Divide 100 by 5 and then by 2.",
        "expected": {"result": 10.0},
        "description": "Testing division tool with sequential division."
    },
    {
        "query": "Subtract 50 from 20.",
        "expected": {"result": -30},
        "description": "Testing subtraction tool with negative results."
    }

]

In [27]:
correct_tasks = []
# Corrected test execution
for index, test in enumerate(test_cases, start=1):
    query = test["query"]
    expected_result = test["expected"]["result"]  # Extract just the value
    
    print(f"\n--- Test Case {index}: {test['description']} ---")
    print(f"Query: {query}")
    
    # Properly format the input
    response = agent.invoke({"messages": [HumanMessage(query)]})
    
    # Find the tool message in the response
    tool_message = None
    for msg in response["messages"]:
        if hasattr(msg, 'name') and msg.name in ['add_numbers_tool', 'substract_numbers_tool', 'multiply_numbers_tool', 'divide_numbers_tool', 'power_numbers_tool']:
            tool_message = msg
            break
    
    if tool_message:
        # Parse the tool result from its content
        import json
        tool_result = json.loads(tool_message.content) 
        if isinstance(tool_result, str):
            tool_result = float(tool_result)
        print(f"Tool Result: {tool_result}")
        print(f"Expected Result: {expected_result}")
        
        if tool_result == expected_result:
            print(f"✅ Test Passed: {test['description']}")
            correct_tasks.append(test["description"])
        else:
            print(f"❌ Test Failed: {test['description']}")
    else:
        print("❌ No tool was called by the agent")

print("\nCorrectly passed tests:", correct_tasks)


--- Test Case 1: Testing subtraction tool with sequential subtraction. ---
Query: Subtract 100, 20, and 10.
Tool Result: 70.0
Expected Result: 70
✅ Test Passed: Testing subtraction tool with sequential subtraction.

--- Test Case 2: Testing multiplication tool for a list of numbers. ---
Query: Multiply 2, 3, and 4.
Tool Result: 24.0
Expected Result: 24
✅ Test Passed: Testing multiplication tool for a list of numbers.

--- Test Case 3: Testing division tool with sequential division. ---
Query: Divide 100 by 5 and then by 2.
Tool Result: 10.0
Expected Result: 10.0
✅ Test Passed: Testing division tool with sequential division.

--- Test Case 4: Testing subtraction tool with negative results. ---
Query: Subtract 50 from 20.
Tool Result: -30.0
Expected Result: -30
✅ Test Passed: Testing subtraction tool with negative results.

Correctly passed tests: ['Testing subtraction tool with sequential subtraction.', 'Testing multiplication tool for a list of numbers.', 'Testing division tool with s

In [13]:
query = "What is the population of Canada? Multiply it by 0.75"

response = agent.invoke({"messages": [(HumanMessage(content=query))]})

print("\nMessage sequence:")
for i, msg in enumerate(response["messages"]):
    print(f"\n--- Message {i+1} ---")
    print(f"Type: {type(msg).__name__}")
    if hasattr(msg, 'content'):
        print(f"Content: {msg.content}")
    if hasattr(msg, 'name'):
        print(f"Name: {msg.name}")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"Tool calls: {msg.tool_calls}")


Message sequence:

--- Message 1 ---
Type: HumanMessage
Content: What is the population of Canada? Multiply it by 0.75
Name: None

--- Message 2 ---
Type: AIMessage
Content: 
Name: None
Tool calls: [{'name': 'search_wikipedia', 'args': {'query': 'population of Canada'}, 'id': 'd69cebc0-b50a-40b5-b29f-443916b88c98', 'type': 'tool_call'}]

--- Message 3 ---
Type: ToolMessage
Content: Page: Population of Canada
Summary: Canada ranks 37th by population among countries of the world, comprising about 0.5% of the world's total, with about 41.5 million Canadians as of 2025. Despite being the second-largest country by total area (fourth-largest by land area), the vast majority of the country is sparsely inhabited, with most of its population south of the 55th parallel north. Just over 60 percent of Canadians live in just two provinces: Ontario and Quebec. Though Canada's overall population density is low, many regions in the south, such as the Quebec City–Windsor Corridor, have population dens